# Phase Gradient Filtering

**Source script:** `grdl/example/image_processing/filtering/phase_gradient.py`

This notebook demonstrates complex-domain spatial filtering on SAR imagery:

1. **ComplexLeeFilter** — adaptive despeckle that preserves interferometric phase
2. **PhaseGradientFilter** — estimates local phase differences (Doppler gradient)
3. **StdDevFilter** — local texture measurement for heterogeneity detection

## Why Phase Gradients Matter

In a single SICD image, the local phase gradient encodes Doppler shifts:
- **Azimuth gradient** → cross-range velocity (moving targets)
- **Range gradient** → terrain slope (interferometric analog)
- **Magnitude** → total phase change rate (edges, discontinuities)

The `ComplexLeeFilter` is applied first because raw SAR phase is corrupted by
speckle — the Lee filter uses local coefficient of variation to adaptively
smooth homogeneous areas while preserving edges and point targets.

## Data Setup

Set `SICD_PATH` in Cell 3. Uses the same SICD file as the sublook notebook.

**Default test path:** `/data/sar/from_duane/SICD/` — Umbra spotlight collects.

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")

In [ ]:
from pathlib import Path
import numpy as np

from grdl.IO import get_reader
from grdl.data_prep import ChipExtractor
from grdl.image_processing.filters import (
    ComplexLeeFilter,
    PhaseGradientFilter,
    StdDevFilter,
)
from grdl.image_processing.intensity import ToDecibels, PercentileStretch

# ════════════════════════════════════════════════════════════════════
# USER CONFIGURATION
# ════════════════════════════════════════════════════════════════════
SICD_PATH = Path("/data/sar/from_duane/SICD/2023-01-23-18-25-33_UMBRA-05_SICD.nitf")
CHIP_SIZE = 2048          # pixels per side
LEE_KERNEL = 7            # ComplexLeeFilter window size
GRAD_KERNEL = 5           # PhaseGradientFilter window size
PLOW = 2.0                # lower percentile for amplitude stretch
PHIGH = 98.0              # upper percentile for amplitude stretch

assert SICD_PATH.exists(), f"SICD file not found: {SICD_PATH}"
print(f"Target: {SICD_PATH.name}")

## 1. Read Complex Chip

Extract a center chip from the SICD. The chip is complex-valued —
both amplitude and phase carry information.

In [ ]:
with get_reader('sicd', SICD_PATH) as reader:
    meta = reader.metadata
    rows, cols = reader.get_shape()
    print(f"Image size: {rows} x {cols}")

    extractor = ChipExtractor(nrows=rows, ncols=cols)
    region = extractor.chip_at_point(
        rows // 2, cols // 2,
        row_width=CHIP_SIZE, col_width=CHIP_SIZE,
    )
    chip = reader.read_chip(
        region.row_start, region.row_end,
        region.col_start, region.col_end,
    )

print(f"Chip shape: {chip.shape}, dtype: {chip.dtype}")
print(f"Amplitude range: [{np.abs(chip).min():.4f}, {np.abs(chip).max():.4f}]")
print(f"Phase range: [{np.angle(chip).min():.4f}, {np.angle(chip).max():.4f}] rad")

## 2. Complex Lee Despeckle

The `ComplexLeeFilter` is an adaptive speckle filter that operates on
complex SAR data. Unlike magnitude-only Lee, it preserves the phase
relationship between pixels — critical for downstream phase gradient estimation.

Algorithm: For each pixel, computes the local coefficient of variation (CV)
within a `kernel_size × kernel_size` window. High CV → edge/target → minimal
smoothing. Low CV → homogeneous clutter → aggressive smoothing.

In [ ]:
clee = ComplexLeeFilter(kernel_size=LEE_KERNEL)
despeckled = clee.apply(chip)

print(f"Input dtype  : {chip.dtype}")
print(f"Output dtype : {despeckled.dtype}")
print(f"Output shape : {despeckled.shape}")

# Compare phase preservation
phase_diff = np.angle(despeckled) - np.angle(chip)
# Wrap to [-pi, pi]
phase_diff = np.arctan2(np.sin(phase_diff), np.cos(phase_diff))
print(f"Phase difference std: {np.std(phase_diff):.4f} rad")
print(f"(lower = better phase preservation)")

## 3. Phase Gradient Estimation

The `PhaseGradientFilter` computes local phase differences:

$$\Delta\phi_{\text{row}}[i,j] = \arg\left( z[i+1,j] \cdot z^*[i,j] \right)$$

averaged over the kernel window. Three modes:
- `direction='row'` → azimuth phase gradient (Doppler shift)
- `direction='col'` → range phase gradient (terrain slope)
- `direction='magnitude'` → $\sqrt{\Delta\phi_{\text{row}}^2 + \Delta\phi_{\text{col}}^2}$

In [ ]:
pgf_row = PhaseGradientFilter(kernel_size=GRAD_KERNEL, direction='row')
pgf_col = PhaseGradientFilter(kernel_size=GRAD_KERNEL, direction='col')
pgf_mag = PhaseGradientFilter(kernel_size=GRAD_KERNEL, direction='magnitude')

grad_row = pgf_row.apply(despeckled)
grad_col = pgf_col.apply(despeckled)
grad_mag = pgf_mag.apply(despeckled)

print(f"Row gradient    : shape={grad_row.shape}, dtype={grad_row.dtype}")
print(f"  range: [{grad_row.min():.4f}, {grad_row.max():.4f}] rad/pixel")
print(f"Col gradient    : shape={grad_col.shape}, dtype={grad_col.dtype}")
print(f"  range: [{grad_col.min():.4f}, {grad_col.max():.4f}] rad/pixel")
print(f"Magnitude       : shape={grad_mag.shape}, dtype={grad_mag.dtype}")
print(f"  range: [{grad_mag.min():.4f}, {grad_mag.max():.4f}] rad/pixel")

## 4. Local Texture (StdDevFilter)

The standard deviation filter computes local variability in amplitude.
High texture → heterogeneous (buildings, vegetation edges).
Low texture → homogeneous (water, bare ground, parking lots).

In [ ]:
std_filter = StdDevFilter(kernel_size=GRAD_KERNEL)
texture = std_filter.apply(despeckled)

print(f"Texture map: shape={texture.shape}, dtype={texture.dtype}")
print(f"  range: [{texture.min():.6f}, {texture.max():.6f}]")
print(f"  mean:  {texture.mean():.6f}")

## 5. Visualization with napari

Display all computed layers:
- Original amplitude (dB stretched)
- Despeckled amplitude
- Phase gradient components (row, col, magnitude)
- Texture map

Use napari's layer panel to toggle visibility and compare.

In [ ]:
import napari

to_db = ToDecibels()
stretch = PercentileStretch(plow=PLOW, phigh=PHIGH)

chip_display = stretch.apply(to_db.apply(chip))
despeckled_display = stretch.apply(to_db.apply(despeckled))

viewer = napari.Viewer(title="Phase Gradient Analysis")

# Amplitude layers
viewer.add_image(chip_display, name="Original (dB)", colormap="gray",
                 contrast_limits=[0, 1])
viewer.add_image(despeckled_display, name="Despeckled (dB)", colormap="gray",
                 contrast_limits=[0, 1], visible=False)

# Phase gradient layers (diverging colormap: blue=negative, red=positive)
viewer.add_image(grad_row, name="Phase Grad (azimuth)", colormap="RdBu_r",
                 contrast_limits=[-0.5, 0.5], visible=False)
viewer.add_image(grad_col, name="Phase Grad (range)", colormap="RdBu_r",
                 contrast_limits=[-0.5, 0.5], visible=False)
viewer.add_image(grad_mag, name="Phase Grad (magnitude)", colormap="hot",
                 contrast_limits=[0, 1.0], visible=False)

# Texture
viewer.add_image(texture, name="Texture (StdDev)", colormap="viridis",
                 visible=False)

print(f"napari viewer opened with {len(viewer.layers)} layers")
print("Toggle layers to compare original, despeckled, gradients, and texture.")

## 6. Statistical Summary

Quantify the despeckle effect and gradient distributions.

In [ ]:
# Speckle reduction: ratio of amplitude standard deviation
amp_orig = np.abs(chip)
amp_desp = np.abs(despeckled)

enl_orig = (amp_orig.mean() / amp_orig.std()) ** 2
enl_desp = (amp_desp.mean() / amp_desp.std()) ** 2

print("═══ Speckle Reduction ═══")
print(f"  ENL (original)   : {enl_orig:.2f}")
print(f"  ENL (despeckled) : {enl_desp:.2f}")
print(f"  Improvement      : {enl_desp/enl_orig:.1f}x")

print("\n═══ Phase Gradient Statistics ═══")
print(f"  Azimuth grad std : {np.std(grad_row):.4f} rad/px")
print(f"  Range grad std   : {np.std(grad_col):.4f} rad/px")
print(f"  Magnitude mean   : {np.mean(grad_mag):.4f} rad/px")
print(f"  Magnitude p99    : {np.percentile(grad_mag, 99):.4f} rad/px")

# Pixels with high phase gradient (potential movers or edges)
threshold = np.percentile(grad_mag, 99)
n_high = np.sum(grad_mag > threshold)
print(f"\n  Pixels > p99 threshold ({threshold:.4f}): {n_high} "
      f"({100*n_high/grad_mag.size:.2f}% of chip)")

---

## Summary

This notebook replaces `grdl/example/image_processing/filtering/phase_gradient.py`.

Key patterns:
- `get_reader('sicd', path)` — factory pattern for format-agnostic IO (replaces direct `SICDReader` import)
- `ComplexLeeFilter` → preserves phase for downstream gradient estimation
- `PhaseGradientFilter(direction=...)` → directional phase change measurement
- `StdDevFilter` → local amplitude variability (texture)
- All filters operate on complex input; outputs are real-valued
- ENL (Equivalent Number of Looks) quantifies speckle reduction effectiveness